# Test BioBERT NER on Real Images

Runs OCR + NER on all 27 real test images in `test_images/` and saves every
prediction to a JSON file for manual review. There is no ground-truth NER
labels for these images, so this notebook does not compute an automatic
accuracy score -- open the saved JSON and check each image's predicted
entities against the actual image by eye.

Re-run this notebook after retraining BioBERT
(`training_notebooks/finetune_biobert_ner.ipynb`) to see how the *new*
model performs on real images, not just its own synthetic test set.

In [2]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from config import TEST_IMAGES_PATH
from models import OCREngine, NERExtractor

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\PRIYANKA\Desktop\MedAi


In [3]:
labels_df = pd.read_csv(PROJECT_ROOT / "datasets" / "test_labels.csv")
true_labels = dict(zip(labels_df["image_name"], labels_df["true_label"]))

image_dir = PROJECT_ROOT / TEST_IMAGES_PATH
image_paths = sorted(image_dir.glob("*"))

print(f"Found {len(image_paths)} test images in {image_dir}")
print(f"Labels loaded for {len(true_labels)} images")

Found 27 test images in C:\Users\PRIYANKA\Desktop\MedAi\test_images
Labels loaded for 27 images


In [4]:
print("Loading OCREngine (PaddleOCR) ...")
ocr_engine = OCREngine()

print("Loading NERExtractor (BioBERT) ...")
ner_extractor = NERExtractor()

print("Both models loaded.")

Loading OCREngine (PaddleOCR) ...


c:\Users\PRIYANKA\Desktop\MedAi\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)


Loading NERExtractor (BioBERT) ...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3608.25it/s]

Both models loaded.


In [5]:
results = {}

for path in image_paths:
    name = path.name
    print(f"Processing {name} ...")
    try:
        ocr_data = ocr_engine.extract_text(str(path))
        ner_data = ner_extractor.extract_entities(ocr_data["raw_text"])

        results[name] = {
            "true_label": true_labels.get(name, "unknown"),
            "ocr_word_count": ocr_data["word_count"],
            "ocr_avg_confidence": ocr_data["avg_confidence"],
            "raw_text": ocr_data["raw_text"],
            "entities": ner_data["entities"],
            "entity_count": ner_data["entity_count"],
            "ner_avg_confidence": ner_data["avg_confidence"],
        }
    except Exception as e:
        results[name] = {"error": str(e)}
        print(f"  FAILED: {e}")

print(f"\nProcessed {len(results)} images")

Processing 001img.jpg ...
Processing 002img.jpg ...
Processing 003img.jpg ...
Processing 004img.jpeg ...
Processing 005img.jpeg ...
Processing 006img.png ...
Processing 007img.jpg ...
Processing 008img.jpeg ...
Processing 009img.jpg ...
Processing 010img.jpg ...
Processing 011img.png ...
Processing 012img.jpg ...
Processing 013img.jpg ...
Processing 014img.jpeg ...
Processing 015img.jpeg ...
Processing 016img.jpg ...
Processing 017img.png ...
Processing 018img.jpg ...
Processing 019img.jpeg ...
Processing 020img.jpg ...
Processing 021img.png ...
Processing 022img.jpeg ...
Processing 023img.jpg ...
Processing 024img.webp ...
Processing 025img.jpeg ...
Processing 026img.jpeg ...
Processing 027img.png ...

Processed 27 images


In [6]:
output_path = Path.cwd() / "test_results" / "ner_real_images_results.json"
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"Saved results to {output_path}")

Saved results to c:\Users\PRIYANKA\Desktop\MedAi\evaluation\test_results\ner_real_images_results.json


In [7]:
print(f"{'Image':20s} {'True Label':12s} {'Entities':9s} {'NER Conf':9s} {'OCR Conf':9s}")
print("-" * 65)
for name, r in results.items():
    if "error" in r:
        print(f"{name:20s} ERROR: {r['error']}")
        continue
    print(f"{name:20s} {r['true_label']:12s} {r['entity_count']:9d} {r['ner_avg_confidence']:8.1f}% {r['ocr_avg_confidence']:8.1f}%")

Image                True Label   Entities  NER Conf  OCR Conf 
-----------------------------------------------------------------
001img.jpg           Prescription        30     69.7%     99.0%
002img.jpg           Non-medical          0      0.0%     98.7%
003img.jpg           Report              57     90.5%     98.9%
004img.jpeg          Report              73     71.6%     97.4%
005img.jpeg          Prescription        91     83.6%     98.7%
006img.png           Non-medical          9     74.2%     99.8%
007img.jpg           Non-medical         19     71.1%     99.7%
008img.jpeg          Prescription        20     66.2%     97.9%
009img.jpg           Prescription        35     82.2%     98.3%
010img.jpg           Report              34     87.5%     97.9%
011img.png           Report               9     78.1%     98.6%
012img.jpg           Non-medical         10     54.9%     89.6%
013img.jpg           Prescription        48     78.2%     98.7%
014img.jpeg          Prescription     